# Module 3: Evaluate

Runs the 50 labelled prompts through your live monitor against both rail configurations, then
compares them. Keep the monitor page open: its run panel fills in while each run proceeds.

Run the cells in order (**Run > Run All Cells** works). Each evaluation takes from a few seconds
(`regex-only`) to two to four minutes (`content-safety`).

## 1. Setup and the dataset

In [ ]:
import json, ssl, urllib.request
import pandas as pd
import matplotlib.pyplot as plt
import labeval as lab  # the evaluation code, next to this notebook (also used by the lab's tests)

NS = lab.namespace()
MONITOR = lab.monitor_url()
rows = lab.load_rows()
print(f"namespace: {NS}\nmonitor:   {MONITOR}")
pd.DataFrame(rows).groupby("label").size().rename("prompts")

In [ ]:
def tick(line):  # one character per prompt: . correct, x wrong, ! no verdict
    outcome, verdict = line.split()[1], line.split()[2]
    print("!" if verdict == "error" else "." if outcome in ("TP", "TN") else "x", end="", flush=True)

def metrics_table(*results):
    return pd.DataFrame([{"run": r["run"], "config": r["config"], "errors as": r["errors_as"],
                          **r["counts"], "errors": r["errors"],
                          **{k: round(v, 2) for k, v in r["metrics"].items()}} for r in results]).set_index("run")

def confusion(ax, res):
    c = res["counts"]
    grid = [[c["TN"], c["FP"]], [c["FN"], c["TP"]]]
    ax.imshow(grid, cmap="Blues")
    for i, row in enumerate(grid):
        for j, v in enumerate(row):
            ax.text(j, i, f"{['TN', 'FP', 'FN', 'TP'][2 * i + j]}\n{v}", ha="center", va="center", fontsize=13,
                    color="white" if v > max(map(max, grid)) / 2 else "black")
    ax.set_xticks([0, 1], ["allowed", "blocked"]); ax.set_yticks([0, 1], ["safe", "unsafe"])
    ax.set_xlabel("guardrails verdict"); ax.set_ylabel("label")
    m = res["metrics"]
    ax.set_title(f"{res['config']}\nP={m['precision']:.2f} R={m['recall']:.2f} FPR={m['fpr']:.2f}")

def wrong(res):
    return pd.DataFrame([r for r in res["results"] if r["outcome"] in ("FP", "FN")],
                        columns=["outcome", "id", "prompt", "rail", "categories"])

## 2. Evaluate `regex-only`

In [ ]:
regex = lab.evaluate("regex-only", rows, progress=tick)
print("\n" + lab.summary(regex))

## 3. Evaluate `content-safety`

In [ ]:
safety = lab.evaluate("content-safety", rows, progress=tick)
print("\n" + lab.summary(safety))

## 4. Compare

In [ ]:
metrics_table(regex, safety)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
confusion(axes[0], regex)
confusion(axes[1], safety)
plt.tight_layout()

## 5. What each configuration got wrong

In [ ]:
wrong(regex)

In [ ]:
wrong(safety)

## 6. Cross-check against the monitor

The monitor counted the same requests independently. Two computations that agree rule out a bug
in either one.

In [ ]:
runs = lab.monitor_runs()
for res in (regex, safety):
    mine = res["counts"]
    theirs = runs[res["run"]]
    agree = all(mine[k] == theirs[k.lower()] for k in ("TP", "FN", "FP", "TN"))
    print(f"{res['run']}: notebook {mine}  monitor {{'TP': {theirs['tp']}, 'FN': {theirs['fn']}, "
          f"'FP': {theirs['fp']}, 'TN': {theirs['tn']}}}  ->  {'AGREE' if agree else 'DISAGREE'}")
    assert agree